In [0]:
%sql
USE CATALOG db_dream_analyst_prod;
CREATE SCHEMA IF NOT EXISTS reports MANAGED LOCATION 'abfss://reports@adlsgen2dreamprod2.dfs.core.windows.net/dream_app/';
USE SCHEMA reports;

In [0]:
%sql
CREATE OR REPLACE TABLE reports.executive_summary
USING DELTA
AS
SELECT
    date_format(t.trip_start_time, 'yyyy/MM') AS report_month,
    COUNT(t.trip_id) AS total_trips,
    SUM(t.fare_amount) AS total_revenue,
    AVG(t.fare_amount) AS average_ticket_size,
    COUNT(DISTINCT t.customer_sk) AS unique_active_customer
FROM gold.fact_trips t
WHERE LOWER(TRIM(t.trip_status)) = 'completed'
GROUP BY 1
ORDER BY 1 DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE reports.report_revenue_by_zone_monthly
USING DELTA
AS
WITH MonthlyZoneRevenue AS (
    SELECT
        date_format(t.trip_start_time, 'yyyy/MM') AS report_month,
        l.city,
        l.state,
        SUM(t.fare_amount) AS total_revenue,
        COUNT(t.trip_id) AS total_trips,
        ROUND(AVG(t.fare_amount),2) AS average_ticket_size
    FROM gold.fact_trips t 
    JOIN gold.dim_locations l ON t.pickup_location_sk = l.location_sk
    WHERE t.trip_status = 'completed'
    GROUP BY 1,2,3
)
SELECT
    report_month, city, state, total_revenue, total_trips, average_ticket_size,
    DENSE_RANK() OVER(PARTITION BY report_month ORDER BY total_revenue DESC) AS zone_revenue_rank
FROM MonthlyZoneRevenue;

In [0]:
%sql
CREATE OR REPLACE TABLE reports.report_revenue_zone_overall
USING DELTA
AS
SELECT
    l.city,
    l.state,
    COUNT(t.trip_id) AS total_trips,
    SUM(t.fare_amount) AS total_revenue,
    AVG(t.fare_amount) AS average_ticket_size,
    RANK() OVER(PARTITION BY l.city ORDER BY SUM(t.fare_amount) DESC) as zone_rank
FROM gold.fact_trips t
JOIN gold.dim_locations l ON t.pickup_location_sk = l.location_sk
WHERE t.trip_status = 'completed'
GROUP BY 1,2
LIMIT 5;

In [0]:
%sql
CREATE OR REPLACE TABLE reports.report_lost_revenue
USING DELTA
AS
SELECT
    l.city,
    t.payment_method,
    COUNT(t.trip_id) AS cancelled_trips,
    SUM(t.fare_amount) AS lost_revenue
FROM gold.fact_trips t
JOIN gold.dim_locations l ON t.pickup_location_sk = l.location_sk
WHERE t.trip_status = 'cancelled'
GROUP BY 1,2
ORDER BY 4 DESC;


In [0]:
%sql
CREATE OR REPLACE TABLE reports.report_customer_vip_list
USING DELTA
AS
SELECT
    c.full_name,
    c.email,
    COUNT(t.trip_id) AS total_trip_taken,
    SUM(t.fare_amount)  AS total_lifetime_spend,
    CASE WHEN SUM(t.fare_amount) > 5000 THEN 'Platinum'
        WHEN SUM(t.fare_amount) > 3000 THEN 'GOLD'
        WHEN SUM(t.fare_amount) > 2000 THEN 'SILVER'
        ELSE 'BRONZE'
    END AS customer_tier
FROM gold.fact_trips t
JOIN gold.dim_customers c ON t.customer_sk = c.customer_sk
WHERE t.trip_status = 'completed'
GROUP BY 1,2;

In [0]:
%sql
CREATE OR REPLACE TABLE reports.report_peak_demand
USING DELTA
AS
SELECT
    HOUR(t.trip_start_time) AS hour_of_day,
    CASE WHEN HOUR(t.trip_start_time) BETWEEN 7 AND 10 THEN 'Morning Rush'
        WHEN HOUR(t.trip_start_time) BETWEEN 16 AND 20 THEN 'Evening Rush'
        ELSE 'off-peak'
    END AS time_category,
    COUNT(t.trip_id) AS total_trips
FROM gold.fact_trips t
GROUP BY 1, 2
ORDER BY 1;

In [0]:
%sql
CREATE OR REPLACE TABLE reports.report_top_travel_corridor
USING DELTA
AS
SELECT
    p.state AS pickup_state,
    d.state AS dropoff_state,
    COUNT(t.trip_id) AS total_trips,
    SUM(t.fare_amount) AS total_revenue,
    ROUND(AVG(t.fare_amount),2) AS average_ticket_size
FROM gold.fact_trips t
JOIN gold.dim_locations p ON t.pickup_location_sk = p.location_sk
JOIN gold.dim_locations d ON t.dropoff_location_sk = d.location_sk
WHERE t.trip_status = 'completed'
GROUP BY 1, 2
ORDER BY 3 DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE reports.report_unit_economic
USING DELTA
AS
SELECT
    CASE WHEN distance_km < 5 THEN 'Short (<5km)'
        WHEN distance_km BETWEEN 5 AND 20 THEN 'Medium (5km-20km)'
        ELSE 'Long (>20km)'
    END AS distance_category,
    COUNT(trip_id) AS total_trip,
    SUM(fare_amount) AS total_revenue,
    SUM(fare_amount * 0.20) AS company_net_revenue,
    SUM(fare_amount) / SUM(distance_km) AS per_km_revenue
FROM gold.fact_trips
WHERE trip_status = 'completed'
GROUP BY 1;

In [0]:
%sql
SELECT DISTINCT trip_status, COUNT(*) 
FROM gold.fact_trips 
GROUP BY 1;